In [1]:
import pandas as pd
import numpy as np

# Load existing features from v1
X = pd.read_csv('data/processed/features.csv')
y = pd.read_csv('data/processed/target.csv').squeeze()

# Load new team stats from WC 2022
team_stats = pd.read_csv('data/processed/team_stats_wc2022.csv')

print("V1 features shape:", X.shape)
print("Team stats shape:", team_stats.shape)
print("\nTeam stats columns:", team_stats.columns.tolist())
print("\nSample team stats:")
team_stats.head()

V1 features shape: (32291, 8)
Team stats shape: (32, 11)

Team stats columns: ['team', 'total_goals', 'total_shots', 'total_xg', 'total_passes', 'total_assists', 'total_yellow_cards', 'total_red_cards', 'top_scorer_goals', 'top_scorer_xg', 'squad_size']

Sample team stats:


,team,total_goals,total_shots,total_xg,total_passes,total_assists,total_yellow_cards,total_red_cards,top_scorer_goals,top_scorer_xg,squad_size
0,Argentina,23,110,20.988443,4615,8,11,0,9,7.598649,24
1,France,18,106,14.955858,3926,12,7,0,9,5.016752,24
2,Croatia,15,87,13.169702,4511,8,5,0,2,2.195553,20
3,England,13,63,8.735856,3238,9,1,0,3,2.330980,21
4,Netherlands,13,48,8.911287,3005,8,4,0,3,1.181856,21


In [2]:
# Load the original results data to get team names per match
results = pd.read_csv('data/raw/results.csv')
results['date'] = pd.to_datetime(results['date'])
results = results.dropna(subset=['home_score', 'away_score'])
results = results.sort_values('date').reset_index(drop=True)

# Filter to post-1990 to match v1
results_modern = results[results['date'].dt.year >= 1990].copy()

print("Results shape:", results_modern.shape)
print("\nWC 2022 teams in our results data:")

# Check which WC 2022 teams match our results data team names
wc_teams = set(team_stats['team'].tolist())
results_teams = set(results_modern['home_team'].tolist())
matches = wc_teams.intersection(results_teams)
missing = wc_teams - results_teams

print(f"Matched: {len(matches)}/32 teams")
print(f"Missing: {missing}")

Results shape: (32291, 9)

WC 2022 teams in our results data:
Matched: 32/32 teams
Missing: set()


In [3]:
# Merge team stats into match results
# Rename columns for home and away team perspective
home_stats = team_stats.add_prefix('home_').rename(columns={'home_team': 'home_team'})
away_stats = team_stats.add_prefix('away_').rename(columns={'away_team': 'away_team'})

# Merge with results
results_modern = results_modern.merge(home_stats, on='home_team', how='left')
results_modern = results_modern.merge(away_stats, on='away_team', how='left')

print("After merge shape:", results_modern.shape)
print("\nNew columns added:", [c for c in results_modern.columns if c.startswith('home_total') or c.startswith('away_total')])

After merge shape: (32291, 29)

New columns added: ['home_total_goals', 'home_total_shots', 'home_total_xg', 'home_total_passes', 'home_total_assists', 'home_total_yellow_cards', 'home_total_red_cards', 'away_total_goals', 'away_total_shots', 'away_total_xg', 'away_total_passes', 'away_total_assists', 'away_total_yellow_cards', 'away_total_red_cards']


In [4]:
# Load elo ratings
import json
with open('data/processed/elo_ratings.json', 'r') as f:
    elo_ratings = json.load(f)

# Add elo and form features (reusing logic from v1)
from src.features import compute_elo_ratings, compute_form

results_modern, _ = compute_elo_ratings(results_modern)
results_modern = compute_form(results_modern)

# Create target
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 2
    elif row['home_score'] < row['away_score']:
        return 0
    else:
        return 1

results_modern['target'] = results_modern.apply(get_result, axis=1)
results_modern['is_wc'] = (results_modern['tournament'] == 'FIFA World Cup').astype(int)
results_modern['neutral'] = results_modern['neutral'].astype(int)

# V2 feature set
v2_features = [
    'home_elo', 'away_elo', 'elo_diff',
    'home_form', 'away_form', 'form_diff',
    'neutral', 'is_wc',
    'home_total_goals', 'away_total_goals',
    'home_total_xg', 'away_total_xg',
    'home_total_shots', 'away_total_shots',
    'home_top_scorer_goals', 'away_top_scorer_goals',
    'home_total_yellow_cards', 'away_total_yellow_cards'
]

X_v2 = results_modern[v2_features].fillna(0)
y_v2 = results_modern['target']

print("V2 feature matrix shape:", X_v2.shape)
print("\nFeature columns:", v2_features)

# Save
X_v2.to_csv('data/processed/features_v2.csv', index=False)
y_v2.to_csv('data/processed/target_v2.csv', index=False)
print("\nSaved!")

V2 feature matrix shape: (32291, 18)

Feature columns: ['home_elo', 'away_elo', 'elo_diff', 'home_form', 'away_form', 'form_diff', 'neutral', 'is_wc', 'home_total_goals', 'away_total_goals', 'home_total_xg', 'away_total_xg', 'home_total_shots', 'away_total_shots', 'home_top_scorer_goals', 'away_top_scorer_goals', 'home_total_yellow_cards', 'away_total_yellow_cards']

Saved!


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, log_loss
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Time-based split
split = int(len(X_v2) * 0.8)
X_train, X_test = X_v2[:split], X_v2[split:]
y_train, y_test = y_v2[:split], y_v2[split:]

# Train all 3 models
models = {
    'Logistic Regression': CalibratedClassifierCV(LogisticRegression(max_iter=1000)),
    'Random Forest': CalibratedClassifierCV(RandomForestClassifier(n_estimators=100, random_state=42)),
    'XGBoost': CalibratedClassifierCV(xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss'))
}

results_v2 = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    acc = accuracy_score(y_test, y_pred)
    ll = log_loss(y_test, y_proba)
    results_v2[name] = {'accuracy': acc, 'log_loss': ll, 'model': model}
    print(f"{name}: Accuracy={acc:.3f}, Log Loss={ll:.3f}")

print("\nV1 baseline: Accuracy=0.605, Log Loss=0.877")
print("Best v2 model:", min(results_v2, key=lambda x: results_v2[x]['log_loss']))

Logistic Regression: Accuracy=0.606, Log Loss=0.881
Random Forest: Accuracy=0.597, Log Loss=0.910
XGBoost: Accuracy=0.601, Log Loss=0.897

V1 baseline: Accuracy=0.605, Log Loss=0.877
Best v2 model: Logistic Regression


In [6]:
wc_competitions = competitions[competitions['competition_name'] == 'FIFA World Cup']
print(wc_competitions[['competition_id', 'season_id', 'season_name']])

NameError: name 'competitions' is not defined

In [7]:
from statsbombpy import sb
import pandas as pd

competitions = sb.competitions()
wc_competitions = competitions[competitions['competition_name'] == 'FIFA World Cup']
print(wc_competitions[['competition_id', 'season_id', 'season_name']])

    competition_id  season_id season_name
30              43        106        2022
31              43          3        2018
32              43         55        1990
33              43         54        1986
34              43         51        1974
35              43        272        1970
36              43        270        1962
37              43        269        1958


In [9]:
import warnings
warnings.filterwarnings('ignore')

wc_seasons = {
    1958: 269,
    1962: 270,
    1970: 272,
    1974: 51,
    1986: 54,
    1990: 55,
    2018: 3,
    2022: 106
}

all_tournament_stats = []

for year, season_id in wc_seasons.items():
    print(f"Processing {year} World Cup...")
    
    matches = sb.matches(competition_id=43, season_id=season_id)
    
    all_events = []
    for match_id in matches['match_id']:
        events = sb.events(match_id=match_id)
        all_events.append(events)
    
    df = pd.concat(all_events, ignore_index=True)
    
    shots = df[df['type'] == 'Shot']
    fouls = df[df['type'] == 'Foul Committed']
    passes = df[df['type'] == 'Pass']
    
    team_goals = shots[shots['shot_outcome'] == 'Goal'].groupby('team').size()
    team_shots = shots.groupby('team').size()
    team_xg = shots.groupby('team')['shot_statsbomb_xg'].sum() if 'shot_statsbomb_xg' in shots.columns else pd.Series(dtype=float)
    team_passes = passes.groupby('team').size()
    
    # Handle missing card column in older tournaments
    if 'foul_committed_card' in fouls.columns:
        team_yellows = fouls[fouls['foul_committed_card'] == 'Yellow Card'].groupby('team').size()
    else:
        team_yellows = pd.Series(dtype=float)
    
    player_goals = shots[shots['shot_outcome'] == 'Goal'].groupby(['team', 'player']).size()
    top_scorer = player_goals.groupby('team').max() if len(player_goals) > 0 else pd.Series(dtype=float)
    
    stats = pd.DataFrame({
        'total_goals': team_goals,
        'total_shots': team_shots,
        'total_xg': team_xg,
        'total_yellow_cards': team_yellows,
        'total_passes': team_passes,
        'top_scorer_goals': top_scorer
    }).fillna(0).reset_index()
    
    stats['year'] = year
    all_tournament_stats.append(stats)
    print(f"  {year}: {len(stats)} teams processed")

tournament_stats = pd.concat(all_tournament_stats, ignore_index=True)
print(f"\nTotal rows: {len(tournament_stats)}")
print(tournament_stats.head(10))

Processing 1958 World Cup...
  1958: 3 teams processed
Processing 1962 World Cup...
  1962: 2 teams processed
Processing 1970 World Cup...
  1970: 7 teams processed
Processing 1974 World Cup...
  1974: 6 teams processed
Processing 1986 World Cup...
  1986: 4 teams processed
Processing 1990 World Cup...
  1990: 2 teams processed
Processing 2018 World Cup...
  2018: 32 teams processed
Processing 2022 World Cup...
  2022: 32 teams processed

Total rows: 88
    index  total_goals  total_shots   total_xg  total_yellow_cards  \
0  Brazil         10.0           67   8.156584                 0.0   
1  France          2.0           15   1.471243                 0.0   
2  Sweden          2.0            6   0.784291                 0.0   
3     NaN          2.0           34   2.359790                 0.0   
4     NaN          0.0            9   0.895518                 1.0   
5     NaN         19.0          151  14.739627                 2.0   
6     NaN          1.0           32   2.417502      

In [10]:
# Fix the team column issue
tournament_stats = tournament_stats.rename(columns={'index': 'team_fix'})

# For rows where team is NaN, use team_fix
tournament_stats['team'] = tournament_stats.apply(
    lambda row: row['team_fix'] if pd.isna(row['team']) else row['team'], axis=1
)

tournament_stats = tournament_stats.drop(columns=['team_fix'])
tournament_stats = tournament_stats.dropna(subset=['team'])

print("Cleaned shape:", tournament_stats.shape)
print("\nYears and team counts:")
print(tournament_stats.groupby('year')['team'].count())
print("\nSample:")
print(tournament_stats[tournament_stats['year'] == 2022].head(5))

Cleaned shape: (88, 8)

Years and team counts:
year
1958     3
1962     2
1970     7
1974     6
1986     4
1990     2
2018    32
2022    32
Name: team, dtype: int64

Sample:
    total_goals  total_shots   total_xg  total_yellow_cards  total_passes  \
56         23.0          110  20.988443                11.0          4615   
57          3.0           26   1.578172                 7.0          1707   
58          1.0           35   3.694493                 5.0          1867   
59         10.0           99  13.636593                 6.0          3196   
60          4.0           28   3.066743                 6.0          1261   

    top_scorer_goals  year       team  
56               9.0  2022  Argentina  
57               1.0  2022  Australia  
58               1.0  2022    Belgium  
59               3.0  2022     Brazil  
60               2.0  2022   Cameroon  


In [11]:
# Save tournament stats
tournament_stats.to_csv('data/processed/tournament_stats_by_year.csv', index=False)
print("Saved!")

# Now let's do time alignment
# For each WC match in our results, look up the PREVIOUS tournament's stats
# Mapping: use stats from the last available WC before that match year

wc_years_available = sorted(tournament_stats['year'].unique())
print("\nWC years with stats:", wc_years_available)

def get_prev_wc_year(match_year):
    """Get the most recent WC year before the match year"""
    prev_years = [y for y in wc_years_available if y < match_year]
    return max(prev_years) if prev_years else None

# Load results
results = pd.read_csv('data/raw/results.csv')
results['date'] = pd.to_datetime(results['date'])
results = results.dropna(subset=['home_score', 'away_score'])
results = results.sort_values('date').reset_index(drop=True)
results_modern = results[results['date'].dt.year >= 1990].copy()
results_modern['match_year'] = results_modern['date'].dt.year
results_modern['prev_wc_year'] = results_modern['match_year'].apply(get_prev_wc_year)

print("\nPrev WC year distribution:")
print(results_modern['prev_wc_year'].value_counts().sort_index())

Saved!

WC years with stats: [np.int64(1958), np.int64(1962), np.int64(1970), np.int64(1974), np.int64(1986), np.int64(1990), np.int64(2018), np.int64(2022)]

Prev WC year distribution:
prev_wc_year
1986      482
1990    24626
2018     3581
2022     3602
Name: count, dtype: int64


In [12]:
# Merge time-aligned stats for home and away teams
home_stats = tournament_stats.add_prefix('home_').rename(columns={'home_year': 'prev_wc_year'})
away_stats = tournament_stats.add_prefix('away_').rename(columns={'away_year': 'prev_wc_year'})

# Merge on team + prev_wc_year
results_modern = results_modern.merge(
    home_stats.rename(columns={'home_team': 'home_team', 'home_year': 'prev_wc_year'}),
    left_on=['home_team', 'prev_wc_year'],
    right_on=['home_team', 'prev_wc_year'],
    how='left'
)

results_modern = results_modern.merge(
    away_stats.rename(columns={'away_team': 'away_team', 'away_year': 'prev_wc_year'}),
    left_on=['away_team', 'prev_wc_year'],
    right_on=['away_team', 'prev_wc_year'],
    how='left'
)

print("Shape after merge:", results_modern.shape)
print("\nNon-null player features:")
print(results_modern[['home_total_goals', 'away_total_goals']].notna().sum())

Shape after merge: (32291, 23)

Non-null player features:
home_total_goals    2052
away_total_goals    1676
dtype: int64


In [13]:
import json
from src.features import compute_elo_ratings, compute_form

# Keep only rows where we have player stats for BOTH teams
results_aligned = results_modern.dropna(subset=['home_total_goals', 'away_total_goals']).copy()
print("Matches with full player stats:", len(results_aligned))
print("Year distribution:")
print(results_aligned['match_year'].value_counts().sort_index())

Matches with full player stats: 528
Year distribution:
match_year
1990      3
1991      3
1993      2
1994      1
1995      2
1998      1
1999      3
2000      1
2001      1
2004      2
2005      2
2006      1
2007      1
2008      1
2009      1
2010      1
2011      2
2012      3
2014      1
2015      1
2016      1
2017      1
2018      1
2019     54
2020     48
2021     71
2022    108
2023     44
2024     79
2025     57
2026     31
Name: count, dtype: int64


In [14]:
# Post 2018 WC matches with player stats
post_2018 = results_aligned[results_aligned['match_year'] >= 2019]
print("Post-2018 matches with player stats:", len(post_2018))
print("\nTeams represented:")
print(post_2018['home_team'].nunique(), "unique home teams")

Post-2018 matches with player stats: 492

Teams represented:
38 unique home teams


In [15]:
# Add elo and form to the aligned dataset
results_aligned_post2018 = post_2018.copy()

# Add target
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 2
    elif row['home_score'] < row['away_score']:
        return 0
    else:
        return 1

results_aligned_post2018['target'] = results_aligned_post2018.apply(get_result, axis=1)
results_aligned_post2018['is_wc'] = (results_aligned_post2018['tournament'] == 'FIFA World Cup').astype(int)
results_aligned_post2018['neutral'] = results_aligned_post2018['neutral'].astype(int)

# Load elo ratings
with open('data/processed/elo_ratings.json', 'r') as f:
    elo_ratings = json.load(f)

# Add elo diff manually
results_aligned_post2018['home_elo'] = results_aligned_post2018['home_team'].map(lambda t: elo_ratings.get(t, 1000))
results_aligned_post2018['away_elo'] = results_aligned_post2018['away_team'].map(lambda t: elo_ratings.get(t, 1000))
results_aligned_post2018['elo_diff'] = results_aligned_post2018['home_elo'] - results_aligned_post2018['away_elo']

# V2 features
v2_features = [
    'home_elo', 'away_elo', 'elo_diff',
    'neutral', 'is_wc',
    'home_total_goals', 'away_total_goals',
    'home_total_xg', 'away_total_xg',
    'home_total_shots', 'away_total_shots',
    'home_top_scorer_goals', 'away_top_scorer_goals',
    'home_total_yellow_cards', 'away_total_yellow_cards'
]

X_v2 = results_aligned_post2018[v2_features].fillna(0)
y_v2 = results_aligned_post2018['target']

print("V2 feature matrix shape:", X_v2.shape)
print("\nTarget distribution:")
print(y_v2.value_counts().rename({0:'Away Win', 1:'Draw', 2:'Home Win'}))

V2 feature matrix shape: (492, 15)

Target distribution:
target
Home Win    222
Draw        138
Away Win    132
Name: count, dtype: int64


In [16]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, log_loss
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# Use cross-validation since dataset is small (492 rows)
# 5-fold CV is more reliable than a single train/test split here
models = {
    'Logistic Regression': CalibratedClassifierCV(LogisticRegression(max_iter=1000)),
    'Random Forest': CalibratedClassifierCV(RandomForestClassifier(n_estimators=100, random_state=42)),
    'XGBoost': CalibratedClassifierCV(xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss'))
}

print("5-Fold Cross Validation Results:")
print("-" * 45)

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_v2, y_v2, cv=5, scoring='accuracy')
    cv_results[name] = scores.mean()
    print(f"{name}: {scores.mean():.3f} (+/- {scores.std():.3f})")

print("\nV1 baseline (single train/test): 0.605")
best = max(cv_results, key=cv_results.get)
print(f"Best v2 model: {best} ({cv_results[best]:.3f})")

5-Fold Cross Validation Results:
---------------------------------------------
Logistic Regression: 0.534 (+/- 0.044)
Random Forest: 0.473 (+/- 0.021)
XGBoost: 0.484 (+/- 0.030)

V1 baseline (single train/test): 0.605
Best v2 model: Logistic Regression (0.534)
